# INGEST DATA FROM SOURCE TO VECTOR DB

### LOAD SOURCE

In [5]:
from langchain_unstructured import UnstructuredLoader

n = int(input("Enter the number of files : "))
list = []
while ( 0 < n ):
    list.append(input("Enter the path of file :"))
    n-=1

file_paths = list

loader = UnstructuredLoader(file_paths)

docs = loader.load()

print("source loaded")


source loaded
page_content='G.L. BAJAJ' metadata={'source': '/Users/singhaman4545/Downloads/Training_and_Placement_Policy (1).pdf', 'coordinates': {'points': ((253.6298, 126.00490000000002), (253.6298, 142.00490000000002), (341.6458, 142.00490000000002), (341.6458, 126.00490000000002)), 'system': 'PixelSpace', 'layout_width': 595.2756, 'layout_height': 841.8898}, 'file_directory': '/Users/singhaman4545/Downloads', 'filename': 'Training_and_Placement_Policy (1).pdf', 'last_modified': '2026-07-26T23:36:24', 'page_number': 1, 'languages': ['eng'], 'filetype': 'application/pdf', 'category': 'Title', 'element_id': '5fae778ffd46c408ee1c879b648c96e8'}


### SPLIT THE SOURCE

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, 
    chunk_overlap=50
    )

# chunks = text_splitter.split_text(docs)  this is for text content 
chunks = text_splitter.split_documents(docs) # use split_documents for docs


l = len(chunks)

print(l)
while(0<l):
    print(chunks[l-1])
    l-=1


127
page_content='G.L. Bajaj Institute of Technology & Management Plot No. 2, Knowledge Park - III, Greater Noida - 201306 (U.P.)' metadata={'source': '/Users/singhaman4545/Downloads/Training_and_Placement_Policy (1).pdf', 'coordinates': {'points': ((172.0833, 480.5559), (172.0833, 501.5559), (423.19230000000005, 501.5559), (423.19230000000005, 480.5559)), 'system': 'PixelSpace', 'layout_width': 595.2756, 'layout_height': 841.8898}, 'file_directory': '/Users/singhaman4545/Downloads', 'filename': 'Training_and_Placement_Policy (1).pdf', 'last_modified': '2026-07-26T23:36:24', 'page_number': 8, 'languages': ['eng'], 'filetype': 'application/pdf', 'parent_id': 'bbc17c528e82f68b4890c625126ae766', 'category': 'NarrativeText', 'element_id': '2c779b009623f1b74a9a302567a48ebe'}
page_content='Registrar' metadata={'source': '/Users/singhaman4545/Downloads/Training_and_Placement_Policy (1).pdf', 'coordinates': {'points': ((68.3622, 468.7629), (68.3622, 478.7629), (108.9222, 478.7629), (108.9222, 

### CHUNK TO EMEBBEDING

In [17]:
import getpass
import os

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")


In [18]:
from langchain_openai import OpenAIEmbeddings

embedder = OpenAIEmbeddings(
    model="text-embedding-3-small",
    # dimensions=1024
)

#this is the openai embedder which embedd chunks

this will use the embedder to convert the chunk to embedding and then store to qdrantDB

In [ ]:
from langchain_qdrant import QdrantVectorStore

qdrant = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embedder,
    collection_name="Placement Policy",
    url="http://localhost:6333", # (QdrantClient) url of qdrant docker container 
)

print("CHUNKS SUCCESSFULLY CONVERTED AND UPLOADED TO QDRANT")


INFO: HTTP Request: GET http://localhost:6333/collections/Placement%20Policy/exists "HTTP/1.1 200 OK"
INFO: HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO: HTTP Request: PUT http://localhost:6333/collections/Placement%20Policy "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO: HTTP Request: PUT http://localhost:6333/collections/Placement%20Policy/points?wait=true "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO: HTTP Request: PUT http://localhost:6333/collections/Placement%20Policy/points?wait=true "HTTP/1.1 200 OK"


CHUNKS SUCCESSFULLY CONVERTED AND UPLOADED TO QDRANT


Putting the document data to the vector database is complete